# 章节实践

## 概述

通过本章的学习，你已经掌握了功能调试（6.2：`asc.printf`/`asc.dump_tensor`与六步调试法）和性能调优（6.3：msprof op上板采集、CSV瓶颈分析、双缓冲流水优化）。本节是综合实践：给定一个**同时含一个功能bug与一处性能劣化**的Mul算子，请独立完成"**先修复、再提速**"的完整闭环。

**算子要求：**

- 数学表达式：`z = x * y`（逐元素乘法，双目算子）
- dtype float32，shape `(8, 8192)`，format ND
- 核函数命名`vmul_kernel`
- 载体文件：下方TODO模板 `Sources/06.04/mul_buggy_slow.py`

### 开发步骤

| 步骤 | 内容 | 参考小节 |
| --- | --- | --- |
| 1 | 运行TODO模板，复现故障（allclose失败） | 6.2 |
| 2 | 加`asc.printf`/`asc.dump_tensor`定位功能bug | 6.2 |
| 3 | 修复功能bug并验证（输出`Sample mul run success.`） | 6.2 |
| 4 | 用msprof op采集baseline性能数据 | 6.3 |
| 5 | 将串行单buffer改造为TQue双缓冲流水 | 6.3 |
| 6 | 复测对比，确认aiv_time明显下降 | 6.3 |

**问题明细**（两处，均已用`# TODO`标注在模板中）：

1. **功能bug**：Compute阶段使用的算子接口不对——本算子是乘法，模板里却是另一个双目接口，导致输出`z`恰好等于`x + y`；
2. **性能劣化**：`BUFFER_NUM=1、TILE_NUM=1`串行执行，无流水重叠。

In [ ]:
# 环境初始化
import os, subprocess
env = subprocess.check_output("bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True)
for line in env.splitlines():
    if "=" in line: os.environ.__setitem__(*line.split("=", 1))
os.makedirs("Sources/06.04", exist_ok=True)
print("环境初始化完成")

### TODO模板

下方代码是待修复的TODO模板。脚手架已含完整Host侧（launch/验证/main），你需要完成两处`# TODO`：

- **TODO(1)**：排查并修正Compute阶段的算子接口（建议先用6.2的调试方法取证再改）；
- **TODO(2)**：将手动串行流水改造为TQue双缓冲流水（参照6.3的`add_framework.py`结构，含`BUFFER_NUM`/`TILE_NUM`常量调整）。

> 提示：可以分两轮完成——先只改TODO(1)跑通功能，再做TODO(2)提速，对应步骤表的1-3与4-6。

In [ ]:
%%writefile Sources/06.04/mul_buggy_slow.py
# Copyright (c) 2025 Huawei Technologies Co., Ltd.
# This program is free software, you can redistribute it and/or modify it under the terms and conditions of
# CANN Open Software License Agreement Version 2.0 (the "License").
# Please refer to the License for details. You may not use this file except in compliance with the License.
# THIS SOFTWARE IS PROVIDED ON AN "AS IS" BASIS, WITHOUT WARRANTIES OF ANY KIND, EITHER EXPRESS OR IMPLIED,
# INCLUDING BUT NOT LIMITED TO NON-INFRINGEMENT, MERCHANTABILITY, OR FITNESS FOR A PARTICULAR PURPOSE.
# See LICENSE in the root of the software repository for the full text of the License.
# 本文件为第六章章节实践载体：Mul算子（含一个功能bug与一处性能劣化，均需学员修复）

import logging
import argparse
import torch
try:
    import torch_npu
except ModuleNotFoundError:
    pass

import asc
import asc.runtime.config as config
import asc.lib.runtime as rt

USE_CORE_NUM = 8
BUFFER_NUM = 1  # 劣化：单缓冲，无流水重叠
TILE_NUM = 1    # 劣化：整块一次搬运计算，不切tile


logging.basicConfig(level=logging.INFO)


@asc.jit(always_compile=True)
def vmul_kernel(x: asc.GlobalAddress, y: asc.GlobalAddress, z: asc.GlobalAddress, block_length: int):

    offset = asc.get_block_idx() * block_length
    x_gm = asc.GlobalTensor()
    y_gm = asc.GlobalTensor()
    z_gm = asc.GlobalTensor()
    x_gm.set_global_buffer(x + offset, block_length)
    y_gm.set_global_buffer(y + offset, block_length)
    z_gm.set_global_buffer(z + offset, block_length)

    tile_length = block_length // TILE_NUM // BUFFER_NUM

    data_type = x.dtype
    buffer_size = tile_length * BUFFER_NUM * data_type.sizeof()

    # Init a Tensor based on the specified logical position/address/length
    x_local = asc.LocalTensor(data_type, asc.TPosition.VECIN, 0, tile_length * BUFFER_NUM)
    y_local = asc.LocalTensor(data_type, asc.TPosition.VECIN, buffer_size, tile_length * BUFFER_NUM)
    z_local = asc.LocalTensor(data_type, asc.TPosition.VECOUT, buffer_size + buffer_size, tile_length * BUFFER_NUM)

    # TODO(2): 将手动串行流水改造为TQue双缓冲（参考6.3的add_framework.py：
    #   用asc.TPipe/asc.TQue与alloc_tensor/enque/deque/free_tensor队列语义
    #   替代手动LocalTensor与set_flag/wait_flag同步）
    for i in range(TILE_NUM * BUFFER_NUM):
        buf_id = i % BUFFER_NUM

        # Operator[] return a new LocalTensor/GlobalTensor with offset from the original starting address
        asc.data_copy(x_local[buf_id * tile_length:], x_gm[i * tile_length:], tile_length)
        asc.data_copy(y_local[buf_id * tile_length:], y_gm[i * tile_length:], tile_length)

        # Synchronization instructions between different pipelines in the same core
        asc.set_flag(asc.HardEvent.MTE2_V, buf_id)
        asc.wait_flag(asc.HardEvent.MTE2_V, buf_id)

        # TODO(1): 排查Compute阶段使用的算子接口是否正确（本算子应为逐元素乘法）
        asc.add(z_local[buf_id * tile_length:], x_local[buf_id * tile_length:], y_local[buf_id * tile_length:],
                tile_length)

        asc.set_flag(asc.HardEvent.V_MTE3, buf_id)
        asc.wait_flag(asc.HardEvent.V_MTE3, buf_id)

        asc.data_copy(z_gm[i * tile_length:], z_local[buf_id * tile_length:], tile_length)

        asc.set_flag(asc.HardEvent.MTE3_MTE2, buf_id)
        asc.wait_flag(asc.HardEvent.MTE3_MTE2, buf_id)


def vmul_launch(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    z = torch.zeros_like(x)

    total_length = z.numel()
    # 本样例采用固定等分切分（每核 block_length、每核内 TILE_NUM * BUFFER_NUM 个等长 tile），
    # 不含尾块边界处理，因此要求 total_length 能被 USE_CORE_NUM * TILE_NUM * BUFFER_NUM 整除，
    # 否则会出现最后一核越界访问或余数数据漏算。处理任意长度需额外实现尾块的有效长度与边界处理。
    align = USE_CORE_NUM * TILE_NUM * BUFFER_NUM
    if total_length % align != 0:
        raise ValueError(
            f"total_length({total_length}) 必须能被 USE_CORE_NUM * TILE_NUM * BUFFER_NUM"
            f"({USE_CORE_NUM} * {TILE_NUM} * {BUFFER_NUM} = {align}) 整除，"
            f"本样例暂不支持非整除长度的尾块处理。"
        )
    block_length = total_length // USE_CORE_NUM

    vmul_kernel[USE_CORE_NUM, rt.current_stream()](x, y, z, block_length)
    return z


def vmul_custom(backend: config.Backend, platform: config.Platform):
    config.set_platform(backend, platform)
    device = "npu" if config.Backend(backend) == config.Backend.NPU else "cpu"
    size = 8 * 8192
    x = torch.rand(size, dtype=torch.float32, device=device)
    y = torch.rand(size, dtype=torch.float32, device=device)
    z = vmul_launch(x, y)
    assert torch.allclose(z, x * y)


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("-r", type=str, default="NPU", help="backend to run")
    parser.add_argument("-v", type=str, default=None, help="platform to run")
    args = parser.parse_args()
    backend = args.r
    platform = args.v
    if backend not in config.Backend.__members__:
        raise ValueError("Unsupported Backend! Supported: ['Model', 'NPU']")
    backend = config.Backend(backend)
    if platform is not None:
        platform_values = [platform.value for platform in config.Platform]
        if platform not in platform_values:
            raise ValueError(f"Unsupported Platform! Supported: {platform_values}")
        platform = config.Platform(platform)
    logging.info("[INFO] start process sample mul.")
    vmul_custom(backend, platform)
    logging.info("[INFO] Sample mul run success.")


### 运行验证

修复并优化完成后，运行下方命令验证。两处问题均解决时输出`[INFO] Sample mul run success.`：

性能复测（步骤4/6，参照6.3）：

```shell
msprof op --output=./prof_mul_baseline python3 Sources/06.04/mul_buggy_slow.py -r NPU   # 修复未优化版
msprof op --output=./prof_mul_fast python3 Sources/06.04/mul_buggy_slow.py -r NPU       # 优化后版本
```

用pandas分别读取两目录下`*/PipeUtilization.csv`对比aiv_time。

In [ ]:
!python3 Sources/06.04/mul_buggy_slow.py -r NPU

### 参考答案

如果你在实现过程中遇到困难，可以查看下方参考答案（修复`asc.mul`误用 + TQue双缓冲优化版）。建议先独立完成两轮调试调优，再对照答案修正。

In [ ]:
!cat ./answer/06.04_chapter_practice/mul_fixed_fast.py